# Fee Chart Code

- OS meaning operating system
- GetCWD meaning Working Directory
- Telling us where Pythons pulling files from

In [105]:
import os

In [106]:
os.listdir("/home/9a8ae02a-2a7d-4aaa-a79e-235075eedfa6/Fee Chart Code")
path = "EXAMPLE.csv"   # ← example, USE EXACT NAME YOU SEE

In [107]:
import pandas as pd

data = pd.read_csv("EXAMPLE.csv", encoding="cp1252")

- Pandas is main package used for tables of data
- Insert file path for csv
- Import csv file and turn it into a table
- Cp1252 reads the file in Windows terms instead of Excels

In [108]:
data.head()

,Project Detail,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- Open: open file so I can see inside it
- "r" meaning read the file
- Errors="ignore" meaning if there files that Python can't interpret, skip them

In [109]:
with open(path, "r", encoding="cp1252", errors="ignore") as f:
    lines = [ln.strip() for ln in f.readlines()]

lines[:20] 

['Project Detail,,,,,,,',
 ',,,,,,,',
 ',,,,,,,',
 ',,,,,,,',
 ',,,,,,,',
 ',,,,,,,',
 ',,,,,,,',
 ',,,,,,,',
 ',,,,,,,',
 'Estimate Overhead,,,"Total',
 'Hours",Billing,,,',
 'Project Number: R211338.01 Callaway New Pool Design and Constructio,,,,,,,',
 'Phase Number: 300FD Final Design (Architecture),,,,,,,',
 '"Task Number: 001 Design, Meetings, Corrections, Specifica",,,,,,,',
 'Labor,,,,,,,',
 '"00039   F      10199  Divis, Sarah    2/14/2023 ",,,0.25,27.5,,,',
 'Phase Number: 300S3 Final Design (Alvine),,,,,,,',
 'Expenses,,,,,,,',
 'Reimbursable Expenses,,,,,,,',
 '53500000 Outside Services (reimb) ,,,,,,,']

In [110]:
# import Pythons regular expression module to search for patterns
import re
phase = None
rows = []

time_re  = re.compile(r"([A-Za-z'\-]+,\s*[A-Za-z'\-]+)\s+(\d{1,2}/\d{1,2}/\d{4}).*?(\d+\.\d+|\.\d+)")
phase_re = re.compile(r"Phase Number:\s*(.*)")

- Re.compile: creates a search pattern for Python in Excel
- "[A-Za-z'\-]+,\s*[A-Za-z'\-]+)": last name, first name
- (\d{1,2}/\d{1,2}/\d{4})): date
- (\d+\.\d+|\.\d+)): Time in hours
- VERBOSE: Allows readable formatting
- (.*): Captures any phase number
- .search: looks for phase numebr anywhere in the line
- .srip(): removes extra spaces
- float(): converts 0.25 to .25

In [112]:
# develop pattern to find the name and hours of time using _re for search
name_date_re = re.compile(r"([A-Za-z'\-]+,\s*[A-Za-z'\-]+)\s+(\d{1,2}/\d{1,2}/\d{4})")
num_re = re.compile(r"[-+]?\d*\.?\d+")

# phase will update as phython sifts through data
rows = []
phase = None

# sift through the report using enumerate
for i, ln in enumerate(lines):
# detect and store the current phase
    mphase = phase_re.search(ln)
    if mphase:
        phase = mphase.group(1).replace(",", " ").strip()
        phase = " ".join(phase.split())
        continue
# detect line that could have time entry (if line doesnt contain name and date, ignore)
    md = name_date_re.search(ln)
    if not md:
        continue
# extract the name and date by groups()
    name, date = md.groups()

# take everything AFTER the date and extract numbers
    after_date = ln.split(date, 1)[1]
    nums = [float(x) for x in num_re.findall(after_date)]

    # choose the number that looks like "hours"
    hours_candidates = [x for x in nums if -24 <= x <= 24]
    if not hours_candidates:
        continue
# pull the first number from that row
    hours = hours_candidates[0]  

# look one line below the index and use replace to get ride of extra characters
    j = i + 1
    def clean_line(s): return s.replace(",", "").replace('"', "").strip()
    while j < len(lines) and clean_line(lines[j]) == "":
        j += 1
    task = clean_line(lines[j]) if j < len(lines) else ""
# save one clean record of compiled data (compd)
    rows.append({"Phase": phase, "Name": name.strip(), "Date": date.strip(), "Hours": hours, "Task": task})

compd = pd.DataFrame(rows)

In [113]:
# convert hour values to numeric values
def to_float_hours(s):
    s = s.strip()
    if s.startswith("(") and s.endswith(")"):
        return -float(s[1:-1])
    return float(s)

### Fact check the total hours and ensure that they match the csv

In [114]:
# sum all time entries to ensure code is working correctly to compare with csv summed time entries 
python_total = compd["Hours"].sum()
python_total

104.5

In [115]:
compd.groupby(["Phase", "Name"])["Hours"].sum()

Phase                                         Name             
300FD Final Design (Architecture)             Divis, Sarah          0.25
302FD Final Design (Electrical)               Davis, Susan         66.50
303FD Final Design (WIG)                      Divis, Sarah          7.00
                                              Pennekamp, Andrew     5.00
305FD Final Design (Survey)                   Divis, Sarah          1.50
307FD Final Design (Transportation)           Divis, Sarah          8.00
402BN Bidding and Negotiation (Electrical)    Divis, Sarah          0.50
403BN Bidding and Negotiation (WIG)           Divis, Sarah          2.00
                                              Pennekamp, Andrew     1.00
500CS Construction Services (Architecture)    Divis, Sarah         -1.50
502CS Construction Services (Electrical)      Divis, Sarah          1.25
503CS Construction Services (WIG)             Divis, Sarah          0.75
                                              Pennekamp, And

### Summarize compiled hours for each person

In [116]:
summary = (
    compd.groupby("Phase", as_index=False)["Hours"]
         .sum()
         .rename(columns={"Hours": "Total_Hours"})
         .sort_values("Total_Hours", ascending=False)
)

In [117]:
summ = (
    compd.groupby("Name", as_index=False)["Hours"]
        .sum()
)
summ

,Name,Hours
0,"Davis, Susan",66.5
1,"Divis, Sarah",30.5
2,"Pennekamp, Andrew",7.5


In [118]:
person_phase = (
    compd.groupby(["Phase", "Name"], as_index=False)["Hours"]
         .sum()
         .rename(columns={"Hours": "Total_Hours"})
         .sort_values(["Phase", "Total_Hours"], ascending=[True, False])
)
person_phase

,Phase,Name,Total_Hours
0,300FD Final Design (Architecture),"Divis, Sarah",0.25
1,302FD Final Design (Electrical),"Davis, Susan",66.50
2,303FD Final Design (WIG),"Divis, Sarah",7.00
3,303FD Final Design (WIG),"Pennekamp, Andrew",5.00
4,305FD Final Design (Survey),"Divis, Sarah",1.50
5,307FD Final Design (Transportation),"Divis, Sarah",8.00
6,402BN Bidding and Negotiation (Electrical),"Divis, Sarah",0.50
7,403BN Bidding and Negotiation (WIG),"Divis, Sarah",2.00
8,403BN Bidding and Negotiation (WIG),"Pennekamp, Andrew",1.00
9,500CS Construction Services (Architecture),"Divis, Sarah",-1.50


In [119]:
hours_by_phase = compd.groupby("Phase")["Hours"].sum()
hours_by_phase

Phase
300FD Final Design (Architecture)                0.25
302FD Final Design (Electrical)                 66.50
303FD Final Design (WIG)                        12.00
305FD Final Design (Survey)                      1.50
307FD Final Design (Transportation)              8.00
402BN Bidding and Negotiation (Electrical)       0.50
403BN Bidding and Negotiation (WIG)              3.00
500CS Construction Services (Architecture)      -1.50
502CS Construction Services (Electrical)         1.25
503CS Construction Services (WIG)                2.25
505CS Construction Services (Survey)             6.00
507CS Construction Services (Transportation)     0.25
602RP RPR Services (Electrical)                  0.00
603RP RPR Services (WIG)                         3.00
604RP RPR Services (Construction)                1.50
Name: Hours, dtype: float64

### Add in staff directory csv to indentify total hours per title

In [120]:
os.listdir("/home/9a8ae02a-2a7d-4aaa-a79e-235075eedfa6/Fee Chart Code")
path = "staffdierectory.csv"   # ← example, USE EXACT NAME YOU SEE

In [121]:
import pandas as pd

titles = pd.read_csv("staffdierectory.csv", encoding="cp1252")

In [122]:
titles.head()

,Name,Title,Notes,Unnamed: 3,Unnamed: 4
0,"Anderson, Seth",Electrical Dept,environmental,NaN,ElectricaL?? I'm not sure
1,"Arens, Steve",PE,structural pm,NaN,NaN
2,"Baldridge, Anthony",Electrical Dept,NaN,NaN,NaN
3,"Barker, Denise",Admin,NaN,NaN,NaN
4,"Bartja, Robert",IDS,"Visualization team, works on rendering, not in...",NaN,NaN


In [123]:
# Clean column names (just in case)
person_phase.columns = person_phase.columns.str.strip()
titles.columns = titles.columns.str.strip()

# Keep only needed columns from titles
titles_small = titles[["Name", "Title"]].copy()

# Clean join keys
person_phase["Name"] = person_phase["Name"].astype(str).str.strip()
titles_small["Name"] = titles_small["Name"].astype(str).str.strip()

# Merge
person_phase_with_title = person_phase.merge(
    titles_small,
    on="Name",
    how="left"
)

# Fill missing titles
person_phase_with_title["Title"] = person_phase_with_title["Title"].fillna("UNKNOWN / NOT FOUND")

# Total hours by Title
hours_by_title = (
    person_phase_with_title
        .groupby("Title", as_index=False)["Total_Hours"]
        .sum()
        .sort_values("Total_Hours", ascending=False)
)

hours_by_title

,Title,Total_Hours
0,Admin,97.0
1,PM,7.5


In [124]:
# sum all time entries to ensure code is working correctly to compare with csv summed time entries 
total_hours = person_phase_with_title["Total_Hours"].sum()
print("Total Hours:", total_hours)

Total Hours: 104.5


In [125]:
unknown_people = person_phase_with_title[
    person_phase_with_title["Title"] == "UNKNOWN / NOT FOUND"
]

unknown_people.sort_values("Total_Hours", ascending=False)

,Phase,Name,Total_Hours,Title
